## Magnetostatic problem

In [57]:
from ngsolve import *
from netgen.read_gmsh import ReadGmsh
from ngsolve.webgui import Draw
from netgen.csg import *
import math
import pyvista as pv
import numpy as np
from ngsolve.krylovspace import GMRes

mesh_path = '../meshes/coil_box'
output_path = '../output/case1/case1_ngsolve'

# Import geometries
mesh = ReadGmsh(mesh_path + ".msh")

for i in range(1, 3):
    # print(i)
    mesh.SetMaterial(i, f'{i}')

for i in range(1, 13):
    # print(i)
    mesh.SetBCName(i-1, f'{i}')

mesh = Mesh(mesh)

mesh.ngmesh.Save(mesh_path + ".vol")

In [58]:
mesh.ne, mesh.nv, mesh.GetMaterials(), mesh.GetBoundaries()

(69629,
 12090,
 ('1', '2'),
 ('1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12'))

In [59]:
# Define material parameters

# f = 123.5e3  # Frequency in Hz
I_coil = 2191.9 # A
mu0 = 4*math.pi*1e-7
# omega = 2*math.pi*f # f Hz excitation

sigma = {"1": 0.0, "2": 1.0}  # Electric conductivity [S/m]
mu_r = {"1": 1.0, "2": 1.0} # Relative permeability [-]

mu_cf = mu0 * CoefficientFunction([mu_r.get(mat, 1.0) for mat in mesh.GetMaterials()])
sigma_cf = CoefficientFunction([sigma.get(mat, 0.0) for mat in mesh.GetMaterials()])

In [60]:
crosssection = Integrate(1, mesh, definedon=mesh.Boundaries("8"))

print(I_coil/(crosssection))

fespot = H1(mesh, order=1, definedon=mesh.Materials("2"), dirichlet="11")
phi,psi = fespot.TnT()
with TaskManager():
    bfa = BilinearForm(sigma_cf*grad(phi)*grad(psi)*dx).Assemble()
    inv = bfa.mat.Inverse(freedofs=fespot.FreeDofs(), inverse="sparsecholesky")
    lff = LinearForm(I_coil/crosssection*psi*ds("8")).Assemble()
    gfphi = GridFunction(fespot)
    gfphi.vec.data = inv * lff.vec


vtk = VTKOutput(mesh,coefs=[gfphi],names=["sol"],filename=output_path + "electric_potential",subdivision=0)
vtk.Do()



30998147.073655326


'../output/case1/case1_ngsolveelectric_potential'

In [9]:
# fes = HCurl(mesh, order=1, complex=True, dirichlet="VacuumSurface", gradientdomains="Tile|Block|Pipe")
fes = HCurl(mesh, order=1, complex=True, dirichlet="1|2|3|4|5|12", nograds = False)
print ("HCurl dofs:", fes.ndof)
u,v = fes.TnT()

a = BilinearForm(fes, symmetric=True, condense=True)
a += 1/mu_cf*curl(u)*curl(v)*dx+1e-7/mu_cf*u*v*dx

pre = preconditioners.BDDC(a)
f = LinearForm(sigma_cf*grad(gfphi)*v*dx("2"))

A = GridFunction(fes)

# with TaskManager():
#     solvers.BVP(bf=a, lf=f, gf=A, pre=pre, \
#                 solver=solvers.CGSolver, solver_flags={"plotrates": True, "tol" : 1e-6})
    

a.Assemble()
f.Assemble()
# pre = preconditioners.Local(a)
with TaskManager():
    A.vec.data = GMRes(a.mat, f.vec, pre=pre, maxsteps=100, printrates=True, tol= 1e-6)

HCurl dofs: 165218
GMRes iteration 1, residual = 33114145.28001466     
GMRes iteration 2, residual = 18049098.012499746     
GMRes iteration 3, residual = 9757568.360107273     
GMRes iteration 4, residual = 6866386.416437042     
GMRes iteration 5, residual = 3721455.4598610117     
GMRes iteration 6, residual = 2170298.858731208     
GMRes iteration 7, residual = 1204571.4005649101     
GMRes iteration 8, residual = 664757.809138118     
GMRes iteration 9, residual = 381317.11431463854     
GMRes iteration 10, residual = 202588.86234449322     
GMRes iteration 11, residual = 110493.91233234301     
GMRes iteration 12, residual = 61919.57427601373     
GMRes iteration 13, residual = 35852.32070656171     
GMRes iteration 14, residual = 20457.577658162543     
GMRes iteration 15, residual = 11570.378364416008     
GMRes iteration 16, residual = 6775.9486936524     
GMRes iteration 17, residual = 3854.5539467038416     
GMRes iteration 18, residual = 2109.6871942483954     
GMRes itera

In [10]:
B_cf = curl(A)

fes_B = VectorH1(mesh, order=1)
B = GridFunction(fes_B)
B.Set(B_cf.real)

B_norm = Norm(B)

# B_norm = Norm(B_cf)

In [11]:
res = pv.read(mesh_path + ".msh")
points = res.points

B_sol = np.zeros((points.shape[0], 3))

B_norm_sol = np.zeros((points.shape[0], 1))

gfphi_out = np.zeros((points.shape[0], 1))

for i in range(points.shape[0]):
    point = mesh(points[i, 0], points[i, 1], points[i, 2])
    B_sol[i, :] = B(point)
    B_norm_sol[i, :] = B_norm(point)
    gfphi_out[i, :] = gfphi(point)

res["B"] = B_sol
res["Magnetic_flux_density_norm"] = B_norm_sol
res["Electric_potential"] = gfphi_out

res.save(output_path + ".vtu")